# DiffuGroundingDINO — full metrics + visualize how it works

Reads the results of an already-trained run (default: the 3-epoch LoRA run on Kaggle
2xT4) and:
1. Plots loss/mAP curves from `log.txt`, finding the real best epoch itself (does NOT
   trust `checkpoint_best_regular.pth` -- that file is corrupted by a best-tracking bug on
   resume across Kaggle sessions, see the explanation in the log-reading cell).
2. Diagnoses a sane score threshold before drawing anything (lesson from the DiffusionDet
   notebook: never assume `score_thresh=0.5`).
3. GT | Prediction gallery (uses the **full-category** caption -- the real eval protocol:
   "given the image, ask for every category, expect every instance back").
4. **Diffusion trace** -- filmstrip + convergence heatmap across DDIM steps. Different from
   section 3 in that **each run only uses a SINGLE category's caption** (e.g. `"dog ."`),
   because this is the part that most needs to show GroundingDINO's text-conditioned
   nature -- using the same 80-category caption as section 3 here would hide the effect of
   changing the text, and would look just like the DiffusionDet demo (which has no text at
   all and always detects everything).

**Required inputs (Add Data before running):**
- The `objdet` Kaggle Dataset (owner `cryandrrich`) -- same as the training notebook,
  contains the already-extracted `diffu_grounding_dino_weights.zip` +
  `data_coco_for_diffu_gdino.zip`.
- **The Notebook Output of the LoRA training session that already ran** (3 epochs) -- open
  the training notebook's Output tab, Add Data > Notebook Output > pick the right version,
  then edit `CKPT_INPUT_DIR` below to match the path Kaggle generates.

In [ ]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/CryAndRRich/object-detection.git"
BRANCH = "main"

WORK = "/kaggle/working"
REPO = f"{WORK}/object-detection"
CODE = f"{REPO}/diffu_grounding_dino"

# Kaggle auto-extracts each zip into a same-named folder (minus .zip) -- same as the
# training notebook.
WEIGHTS_INPUT_DIR = "/kaggle/input/datasets/cryandrrich/objdet/diffu_grounding_dino_weights"
DATA_INPUT_DIR = "/kaggle/input/datasets/cryandrrich/objdet/data_coco_for_diffu_gdino"

# EDIT THIS: point at the output_dir of the LoRA training session (contains log.txt +
# checkpoint*.pth).
CKPT_INPUT_DIR = "/kaggle/input/<your-notebook-output-name>/output/diffu_run1"

# Must match EXACTLY the value used for that training session -- getting it wrong loads the
# wrong keys, or silently retrains part of the model from scratch with no clear error.
USE_LORA = True
LORA_RANK = 8
LORA_ALPHA = 16

# Number of DDIM steps used everywhere in this notebook -- does not affect the trained
# weights, it is purely an inference-time hyperparameter.
SAMPLING_STEPS = 3

# ---- picking images to illustrate ----
N_IMAGES = 6                       # number of random images for diffusion trace + gallery
MIN_GT = 2                         # COCO-minitrain val is much sparser than CrowdHuman -- keep this low
MAX_GT_PER_IMAGE = 3               # gallery: draw up to this many GT boxes as a title highlight
MAX_CATEGORIES_PER_IMAGE_TRACE = 2 # diffusion trace: demo at most this many categories per image
                                    # (each category = one extra forward pass with its own caption --
                                    # raising this scales runtime linearly)
SEED = 0
DRAW_PROPOSALS = 20                # heatmap background only -- not the full 900 queries

# ---- threshold diagnosis + gallery (use the FULL-CATEGORY caption, matching real eval) ----
N_DIAGNOSE_IMAGES = 100            # val images used to measure the score distribution (no drawing, just numbers)
THRESH_SCENARIOS = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
GALLERY_SCORE_THRESH = None        # None = auto-pick after the diagnosis cell; set by hand to override
N_WORST = 12
MAX_NO_PRED = 4                    # cap on the "zero predictions above threshold" group in the worst-case table

# Drawing font -- keep the SAME candidate list as `diffusiondet_diffusion_trace.ipynb` so
# the two notebooks look consistent (avoids Kaggle missing the DejaVu font and silently
# falling back to the much smaller/uglier `ImageFont.load_default()`).
PANEL_W = 380
JPEG_QUALITY = 92

VIZ = f"{WORK}/viz"
os.makedirs(VIZ, exist_ok=True)

## 1. Check environment + install libraries

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| GPU count:", torch.cuda.device_count())

!pip install -q 'transformers>=4.30,<5' 'scipy>=1.9' 'pycocotools>=2.0.6' 'pandas>=1.5'

## 2. Get the code from GitHub

In [ ]:
import subprocess

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(CODE)
import sys
sys.path.insert(0, CODE)
print("cwd:", os.getcwd())

## 3. Check weights + data + the trained checkpoint

In [ ]:
PRETRAIN = f"{WEIGHTS_INPUT_DIR}/diffu_grounding_dino/groundingdino_swint_ogc.pth"
BERT_DIR = f"{WEIGHTS_INPUT_DIR}/diffu_grounding_dino/bert-base-uncased"
VAL_ANN = f"{DATA_INPUT_DIR}/coco/annotations/instances_val2017.json"
VAL_IMAGES = f"{DATA_INPUT_DIR}/coco/val2017"

for p in (PRETRAIN, BERT_DIR, VAL_ANN, VAL_IMAGES):
    assert os.path.exists(p), f"missing: {p}"
print("weights + val data: OK")

LOG_PATH = f"{CKPT_INPUT_DIR}/log.txt"
assert os.path.exists(LOG_PATH), (
    f"could not find {LOG_PATH} -- check that you Added Data for the right training "
    f"Notebook Output"
)
print("checkpoint input: OK --", CKPT_INPUT_DIR)
print(sorted(os.listdir(CKPT_INPUT_DIR)))

## 4. Read `log.txt` -- find the REAL best epoch

`checkpoint_best_regular.pth` from this Kaggle run **cannot be trusted**: `main.py` writes
`checkpoint.pth` (used for auto-resume) *before* `evaluate()`/`best_map_holder.update()`
finishes for that epoch, so every resume (exactly the 1-epoch-per-session pattern used on
Kaggle) resets the "best" tracker back to the previous epoch's value -- directly visible in
the raw log (`restored best-so-far: best=0.0000 @epoch-1` right after epoch 0 had already
reached 0.4664). The per-epoch mAP numbers in `log.txt` are still correct (each computed
independently); only the choice of which checkpoint counts as "best" is wrong. So this cell
recomputes it from `log.txt` instead of trusting `checkpoint_best_regular.pth`.

In [ ]:
import json
import pandas as pd

rows = []
with open(LOG_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows).set_index("epoch").sort_index()

# test_coco_eval_bbox is a 12-number list in the exact order of COCOeval.stats:
# AP, AP50, AP75, AP_s, AP_m, AP_l, AR@1, AR@10, AR@100, AR_s, AR_m, AR_l
_COCO_COLS = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
              "AR@1", "AR@10", "AR@100", "AR_small", "AR_medium", "AR_large"]
coco_df = pd.DataFrame(df["test_coco_eval_bbox"].tolist(), index=df.index, columns=_COCO_COLS)
df = df.join(coco_df)

best_epoch = int(df["AP"].idxmax())
print(f"real best epoch (recomputed from log.txt): {best_epoch}  (AP={df.loc[best_epoch, 'AP']:.4f})")
print()
print(df[["train_loss", "AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "epoch_time"]]
      .round(4).to_string())

CKPT_EPOCH = best_epoch

## 5. Plot loss + mAP curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(df.index, df["train_loss"], "o-", label="train_loss (per-epoch avg)")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Training loss")
ax1.grid(alpha=0.3)

ax2.plot(df.index, df["AP"], "o-", color="tab:green", label="AP (0.5:0.95)")
ax2.plot(df.index, df["AP50"], "s--", color="tab:orange", alpha=0.7, label="AP50")
ax2.axvline(best_epoch, color="red", ls=":", label=f"best epoch (log.txt) = {best_epoch}")
ax2.set_xlabel("epoch"); ax2.set_ylabel("AP"); ax2.set_title("Validation mAP")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{VIZ}/loss_map_curve.png", dpi=130)
plt.show()
print("Note: if this run had more than one resume between epochs, the 'best-so-far' line")
print("printed in the raw training stdout can be temporarily wrong -- the plot above uses AP")
print("recomputed directly from test_coco_eval_bbox on every log line, so it is not affected")
print("by that bug.")

## 6. Build the model + LoRA + load the best checkpoint

Required order (see the README's LoRA section): **build the model -> `inject_lora` -> only
then `load_state_dict`**. A checkpoint saved by a LoRA session has keys shaped like
`...base.weight`/`lora_A`/`lora_B`; calling `load_state_dict` before the LoRA structure
exists means most keys silently fail to match and stay at their random initialisation.

In [ ]:
import main as train_main
import util.misc as utils
from models import build_model
from models.lora import inject_lora
from util.misc import clean_state_dict

DATASETS_JSON = f"{WORK}/datasets_eval.json"
with open(DATASETS_JSON, "w", encoding="utf-8") as f:
    json.dump({"train": [], "val": [{"root": VAL_IMAGES, "anno": VAL_ANN, "dataset_mode": "coco"}]}, f)

argv = [
    "-c", "config/cfg_odvg_diffusion.py",
    "--datasets", DATASETS_JSON,
    "--options",
    f"text_encoder_type={BERT_DIR!r}",
    f"use_lora={USE_LORA}",
    f"lora_rank={LORA_RANK}",
    f"lora_alpha={LORA_ALPHA}",
    f"diff_sampling_timesteps={SAMPLING_STEPS}",
]
args = train_main.get_args_parser().parse_args(argv)
cfg = train_main.load_config_into_args(args)
utils.setup_distributed(args)  # single process here, no torchrun (inference/visualization only)

with open(args.datasets, encoding="utf-8") as f:
    dataset_meta = json.load(f)
if args.use_coco_eval:
    args.coco_val_path = dataset_meta["val"][0]["anno"]

device = torch.device(args.device if torch.cuda.is_available() else "cpu")
print("device:", device, "| use_lora:", args.use_lora, "| use_diffusion:", args.use_diffusion,
      "| diff_sampling_timesteps:", args.diff_sampling_timesteps)

model, criterion, postprocessors = build_model(args)
model.to(device)

if args.use_lora:
    n_wrapped = inject_lora(model, args.lora_target_prefixes, args.lora_rank, args.lora_alpha, args.lora_dropout)
    print(f"LoRA: injected into {n_wrapped} layers")

CKPT_PATH = f"{CKPT_INPUT_DIR}/checkpoint{CKPT_EPOCH:04d}.pth"
assert os.path.exists(CKPT_PATH), (
    f"could not find {CKPT_PATH} -- check save_checkpoint_interval used for training (the "
    f"config default is 1, so this file should exist for EVERY epoch); if only checkpoint.pth "
    f"(the latest one) exists, set CKPT_EPOCH to the last epoch that actually ran."
)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
result = model.load_state_dict(clean_state_dict(ckpt["model"]), strict=False)
print(f"loaded {CKPT_PATH}")
print(f"missing keys: {len(result.missing_keys)}  unexpected keys: {len(result.unexpected_keys)}")
if result.missing_keys:
    print("  (if this is not just diffusion/buffer modules, double-check USE_LORA/LORA_RANK/LORA_ALPHA)")

model.eval()
postprocessor = postprocessors["bbox"]
catid_to_name = dict(zip(postprocessor.category_ids, postprocessor.cat_list))
FULL_EVAL_CAPTION = None  # set in the next cell, once build_caption is available
print(f"total categories in the val set: {len(postprocessor.cat_list)}")

## 7. Pick random val images with enough GT

In [ ]:
import random

from gdino_datasets import build_dataset
from util.vl_utils import build_caption

dataset_val = build_dataset("val", args=args, datasetinfo=dataset_meta["val"][0])
coco_api = dataset_val.coco  # torchvision.datasets.CocoDetection.coco -- pycocotools COCO

random.seed(SEED)
candidates = []
for idx, image_id in enumerate(dataset_val.ids):
    ann_ids = coco_api.getAnnIds(imgIds=image_id, iscrowd=False)
    if len(ann_ids) >= MIN_GT:
        candidates.append(idx)
random.shuffle(candidates)
picked_indices = candidates[:N_IMAGES]
print(f"{len(candidates)} images have >= {MIN_GT} GT out of {len(dataset_val)} val images -- picked {len(picked_indices)}")

# "Detect every category" caption -- the project's real eval protocol: ask for ALL
# categories at once, expect ALL instances back (unlike referring expression, and unlike
# the DiffusionDet demo which has no text concept at all). Used for the gallery + threshold
# diagnosis cells below.
FULL_EVAL_CAPTION = build_caption(postprocessor.cat_list)
print(f"full-category caption ({len(postprocessor.cat_list)} categories): {FULL_EVAL_CAPTION[:150]}...")

## 8. Diagnose a score threshold before drawing anything

Lesson from the DiffusionDet notebook: picking too high a threshold relative to the real
score distribution leaves many images with `n_pred=0`, so every "worst-case" ranking ends up
full of blank images even though the model still has signal, just below the threshold.
`ContrastiveEmbed` has no low-bias initialisation like DiffusionDet's focal-loss prior=0.01,
so **do not assume** a sane threshold -- measure it directly on `N_DIAGNOSE_IMAGES` val
images (using `FULL_EVAL_CAPTION`, matching the real eval protocol).

In [ ]:
import numpy as np
from util.box_ops import box_cxcywh_to_xyxy

def _iou_matrix(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-6)

@torch.no_grad()
def run_full_caption(idx):
    # Forward one image with the full-category caption -- returns (result, target, file_name, image_id).
    image_t, target = dataset_val[idx]
    samples = utils.nested_tensor_from_tensor_list([image_t]).to(device)
    outputs = model(samples, targets=None, captions=[FULL_EVAL_CAPTION])
    orig_size = target["orig_size"][None].to(device)
    result = postprocessor(outputs, orig_size)[0]
    image_id = dataset_val.ids[idx]
    file_name = coco_api.loadImgs(image_id)[0]["file_name"]
    return {k: v.cpu().numpy() for k, v in result.items()}, target, file_name, image_id

diag_indices = candidates[:N_DIAGNOSE_IMAGES] if len(candidates) >= N_DIAGNOSE_IMAGES else candidates
diag_runs = [run_full_caption(i) for i in diag_indices]

rows = []
for thresh in THRESH_SCENARIOS:
    n_preds, n_matched, n_gt_total = [], 0, 0
    for result, target, _, _ in diag_runs:
        keep = result["scores"] >= thresh
        n_preds.append(int(keep.sum()))
        gt_boxes = target["boxes"].numpy()
        if len(gt_boxes):
            gt_boxes_xyxy = box_cxcywh_to_xyxy(torch.as_tensor(gt_boxes)).numpy()
            h, w = target["orig_size"].tolist()
            gt_boxes_xyxy = gt_boxes_xyxy * np.array([w, h, w, h])
            n_gt_total += len(gt_boxes_xyxy)
            if keep.sum():
                ious = _iou_matrix(gt_boxes_xyxy, result["boxes"][keep])
                n_matched += int((ious.max(axis=1) >= 0.5).sum())
    recall = n_matched / max(n_gt_total, 1)
    rows.append(dict(thresh=thresh, avg_preds_per_image=np.mean(n_preds), recall_iou50=recall))

diag_df = pd.DataFrame(rows)
print(diag_df.round(3).to_string(index=False))
print()
print("Threshold choice: prefer a level where recall is still high (>= about 0.8x the recall")
print("at the lowest threshold) while the number of boxes/image stays low enough for the")
print("worst-case table to stay readable. Edit GALLERY_SCORE_THRESH in the config cell to set")
print("it by hand instead of letting the last line below auto-pick.")

if GALLERY_SCORE_THRESH is None:
    target_recall = diag_df["recall_iou50"].iloc[0] * 0.8
    ok = diag_df[diag_df["recall_iou50"] >= target_recall]
    GALLERY_SCORE_THRESH = float(ok["thresh"].max()) if len(ok) else float(diag_df["thresh"].min())
print(f"GALLERY_SCORE_THRESH = {GALLERY_SCORE_THRESH}")

## 9. GT | Prediction gallery -- stratified worst-case

Boxes are drawn **at the original image resolution** (read straight from the file, not
through the resized/normalized tensor) and only then downscaled per panel -- doing it the
other way round loses thin outlines and shifts coordinates from early rounding. Uses
`FULL_EVAL_CAPTION` (every category at once) -- matching the real eval protocol, unlike the
diffusion trace section below (one category per run).

In [ ]:
from PIL import Image, ImageDraw, ImageFont

C_GT_MISS = (220, 30, 30)     # GT the model missed (red)
C_GT_HIT = (80, 80, 255)      # GT matched by at least one prediction above threshold (blue)
C_PRED_HI = (255, 140, 0)     # prediction above threshold, not matching any GT (orange = false positive)
C_PRED_HI_OK = (40, 200, 90)  # prediction above threshold, matching a GT (green)
C_PRED_LOW = (200, 200, 100)  # prediction BELOW threshold -- drawn faint to distinguish "truly blind" vs "threshold too high"

# Keep the SAME font candidate list + matplotlib fallback as
# `diffusiondet_diffusion_trace.ipynb` -- dropping this fallback means Kaggle missing the
# DejaVu font silently falls back to ImageFont.load_default() (small, ugly, size-invariant)
# with no warning.
def _font(size=14, _cache={}):
    if size in _cache:
        return _cache[size]
    cands = ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
             "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    try:
        from matplotlib import font_manager
        cands += [font_manager.findfont("DejaVu Sans:bold", fallback_to_default=False),
                  font_manager.findfont("DejaVu Sans", fallback_to_default=False)]
    except Exception:
        pass
    for p in cands:
        if p and os.path.isfile(p):
            try:
                _cache[size] = ImageFont.truetype(p, size)
                return _cache[size]
            except Exception:
                pass
    _cache[size] = ImageFont.load_default()
    return _cache[size]

F_TITLE, F_LABEL = _font(15), _font(12)

def _label(d, x, y, text, color, font=F_LABEL):
    # Draw text on a solid-color background -- readable regardless of the underlying image,
    # same as how diffusiondet_diffusion_trace.ipynb annotates IoU/score on each box.
    try:
        l, t, r, b = d.textbbox((0, 0), text, font=font)
        tw, th = r - l, b - t
    except Exception:
        tw, th = len(text) * 7, 12
    y = max(0, y - th - 4)
    d.rectangle([x, y, x + tw + 5, y + th + 5], fill=color)
    d.text((x + 2, y + 1), text, fill=(0, 0, 0), font=font)

# Stroke widths measured in FINAL pixels (after downscaling the panel) -- fixed, independent
# of original image resolution or PANEL_W, matching the DiffusionDet notebook's values so
# both look consistent.
BG_WIDTH = 1
GT_WIDTH = 2
TRACK_WIDTH = 4
DOT_R = 4

def _resize_and_scale(im, max_w):
    W, H = im.size
    if W <= max_w:
        return im, 1.0
    h = max(1, round(H * max_w / W))
    return im.resize((max_w, h), Image.LANCZOS), max_w / W

def _scaled(box, scale):
    return [c * scale for c in box]

# ---- text layout helpers -------------------------------------------------------
# A long dynamic title (it embeds the caption text) drawn on one fixed-height line was
# overlapping the per-panel subtitle row below it whenever the title ran wider than a
# couple of panels -- this wraps it to as many lines as needed and only THEN reserves
# exactly that much header height, so subtitles can never sit on the same row.
_MEASURE = ImageDraw.Draw(Image.new("RGB", (10, 10)))

def _wrap_lines(text, font, max_width):
    words = text.split(" ")
    lines, cur = [], ""
    for w in words:
        trial = (cur + " " + w).strip()
        width = _MEASURE.textbbox((0, 0), trial, font=font)[2]
        if width <= max_width or not cur:
            cur = trial
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)
    return lines

def _line_height(font):
    bbox = _MEASURE.textbbox((0, 0), "Ag", font=font)
    return (bbox[3] - bbox[1]) + 4

def _strip(panels, subtitles, title_text, gap=8):
    # One row of same-size panels, a wrapped title above, one subtitle line per panel
    # below the title -- the title's own height is measured first so it can never collide
    # with the subtitle row underneath it.
    pw, ph = panels[0].size
    n = len(panels)
    canvas_w = pw * n + gap * (n - 1)

    title_lines = _wrap_lines(title_text, F_TITLE, canvas_w - 12)
    title_line_h = _line_height(F_TITLE)
    subtitle_line_h = _line_height(F_LABEL)
    top = 8 + title_line_h * len(title_lines) + 6 + subtitle_line_h + 6

    canvas = Image.new("RGB", (canvas_w, top + ph), (255, 255, 255))
    d = ImageDraw.Draw(canvas)
    y = 8
    for line in title_lines:
        d.text((6, y), line, fill=(0, 0, 0), font=F_TITLE)
        y += title_line_h
    y += 6
    for i, (p, t) in enumerate(zip(panels, subtitles)):
        x = i * (pw + gap)
        canvas.paste(p, (x, top))
        d.text((x + 4, y), t, fill=(0, 0, 0), font=F_LABEL)
    return canvas

def _panel_pair(file_name, gt_boxes_xyxy, pred_boxes, pred_scores, thresh, max_w=PANEL_W):
    base = Image.open(os.path.join(VAL_IMAGES, file_name)).convert("RGB")
    hi = pred_scores >= thresh
    matched_gt = np.zeros(len(gt_boxes_xyxy), dtype=bool)
    matched_pred = np.zeros(len(pred_boxes), dtype=bool)
    if len(gt_boxes_xyxy) and hi.sum():
        ious = _iou_matrix(gt_boxes_xyxy, pred_boxes[hi])
        matched_gt = ious.max(axis=1) >= 0.5 if ious.size else matched_gt
        hi_idx = np.where(hi)[0]
        if ious.size:
            matched_pred[hi_idx[ious.max(axis=0) >= 0.5]] = True

    im_gt, s1 = _resize_and_scale(base.copy(), max_w)
    d1 = ImageDraw.Draw(im_gt)
    for i, box in enumerate(gt_boxes_xyxy):
        color = C_GT_HIT if matched_gt[i] else C_GT_MISS
        d1.rectangle(_scaled(box, s1), outline=color, width=GT_WIDTH)

    im_pred, s2 = _resize_and_scale(base.copy(), max_w)
    d2 = ImageDraw.Draw(im_pred)
    for i, box in enumerate(pred_boxes):
        if pred_scores[i] >= thresh:
            color, width = (C_PRED_HI_OK if matched_pred[i] else C_PRED_HI), TRACK_WIDTH
        else:
            color, width = C_PRED_LOW, BG_WIDTH
        d2.rectangle(_scaled(box, s2), outline=color, width=width)
    return im_gt, im_pred, int(matched_gt.sum()), int(hi.sum())

def _stack_pair(im_gt, im_pred, title):
    gap = 8
    w, h = im_gt.size
    canvas_w = w * 2 + gap

    title_lines = _wrap_lines(title, F_TITLE, canvas_w - 12)
    title_line_h = _line_height(F_TITLE)

    # Wrap each panel's own legend to that panel's width -- the fixed "Pred (orange=...)"
    # string is ~389px wide on its own, a few pixels over a 380px-wide panel, so it was
    # clipped at the right edge instead of wrapping.
    gt_caption_lines = _wrap_lines("GT (red=missed, blue=matched)", F_LABEL, w - 12)
    pred_caption_lines = _wrap_lines(
        "Pred (orange=false positive, green=correct, faint yellow=below threshold)", F_LABEL, w - 12
    )
    caption_line_h = _line_height(F_LABEL)
    n_caption_lines = max(len(gt_caption_lines), len(pred_caption_lines))

    top = 8 + title_line_h * len(title_lines) + 6 + caption_line_h * n_caption_lines + 6

    canvas = Image.new("RGB", (canvas_w, top + h), (255, 255, 255))
    d = ImageDraw.Draw(canvas)
    y = 8
    for line in title_lines:
        d.text((6, y), line, fill=(0, 0, 0), font=F_TITLE)
        y += title_line_h
    y += 6
    canvas.paste(im_gt, (0, top))
    canvas.paste(im_pred, (w + gap, top))
    yy = y
    for line in gt_caption_lines:
        d.text((6, yy), line, fill=(0, 0, 0), font=F_LABEL)
        yy += caption_line_h
    yy = y
    for line in pred_caption_lines:
        d.text((w + gap + 6, yy), line, fill=(0, 0, 0), font=F_LABEL)
        yy += caption_line_h
    return canvas

gallery_runs = [run_full_caption(i) for i in picked_indices]

scored = []
for result, target, file_name, image_id in gallery_runs:
    gt_boxes = target["boxes"].numpy()
    h, w = target["orig_size"].tolist()
    gt_xyxy = box_cxcywh_to_xyxy(torch.as_tensor(gt_boxes)).numpy() * np.array([w, h, w, h]) if len(gt_boxes) else np.zeros((0, 4))
    n_hi = int((result["scores"] >= GALLERY_SCORE_THRESH).sum())
    scored.append((image_id, file_name, gt_xyxy, result, n_hi))

no_pred = [r for r in scored if r[4] == 0]
has_pred = [r for r in scored if r[4] > 0]
worst = (no_pred[:MAX_NO_PRED] + has_pred)[:N_WORST]  # stratified: don't let "0 predictions" dominate

os.makedirs(f"{VIZ}/gallery", exist_ok=True)
for image_id, file_name, gt_xyxy, result, n_hi in worst:
    im_gt, im_pred, n_matched, n_hi2 = _panel_pair(
        file_name, gt_xyxy, result["boxes"], result["scores"], GALLERY_SCORE_THRESH
    )
    title = f"image_id={image_id}  |  caption: full-category  |  GT={len(gt_xyxy)} matched={n_matched}  |  pred>=thresh={n_hi2}"
    canvas = _stack_pair(im_gt, im_pred, title)
    canvas.save(f"{VIZ}/gallery/{image_id}.jpg", quality=JPEG_QUALITY)

print(f"saved {len(worst)} GT|Prediction pairs -> {VIZ}/gallery/")

## 10. View the gallery right here

In [ ]:
from IPython.display import display, Image as IPyImage

for image_id, *_ in worst:
    print(f"--- image_id {image_id} ---")
    display(IPyImage(filename=f"{VIZ}/gallery/{image_id}.jpg"))

## 11. Diffusion trace -- filmstrip + convergence heatmap (one category per run)

`model.ddim_sample(..., return_trajectory=True)` does not expose the raw pre-decode noise
(`x_T`, before it is ever run through the decoder) -- only the decoder's per-step outputs.
To show that frame too (and label everything correctly -- see the note below), this cell
manually re-runs the same loop as `DiffuGroundingDINO.ddim_sample()` using its own public
building blocks (`diffusion.init_latent`, `diffusion.latent_to_boxes`,
`diffusion.latent_to_refpoints`, `model.transformer.decode`, `model._decode_predictions`,
`diffusion.boxes_to_latent`, `diffusion.predict_noise_from_start`, `diffusion.ddim_step`) --
same math, verified line-for-line against `models/diffu_groundingdino.py`, just with an
extra frame recorded before the loop starts.

**Why the previous version mislabeled things:** every entry coming out of
`return_trajectory=True` is already a *decoder output* -- even the very first one, computed
from a still-very-noisy reference point, is the model's actual prediction, not literal raw
noise. Labeling it "z_T (pure noise)" was wrong and made it impossible to tell a genuinely
undertrained/noisy first guess apart from what raw, un-decoded Gaussian noise looks like.
Now `z_T` is its own explicit frame (the raw noise, mapped through the box-space clamp only,
never touching the decoder), and every following panel is honestly labeled "step k".

That clamp is also the answer to "why do dots sit near the image edge": `init_latent`
draws `x ~ N(0, I)` in an unconstrained latent space, and turning that into a box requires
clamping into `[box_eps, 1-box_eps]` (`RefPointDiffusion.latent_to_boxes`) -- a Gaussian's
tails routinely land outside `[0, 1]` once rescaled, and those get clamped straight to the
image border. So the `z_T` panel showing points hugging the edges is *expected DDPM
behaviour*, not a rendering bug. If the following `step 1/2/3` panels *also* keep drifting
toward the edges instead of visibly pulling in toward real objects, that is a genuine
observation about this checkpoint (only 3 LoRA epochs) rather than a display bug -- the
panels below now make it possible to tell the two apart.

**Different from the gallery section above:** each run here uses a caption for a **single
category** (e.g. `"dog ."`), not all 80 categories -- the part that most needs to show
GroundingDINO's text-conditioned nature. For each image, up to
`MAX_CATEGORIES_PER_IMAGE_TRACE` distinct categories present in its GT are picked, a separate
DDIM run is done per category, and the caption used is printed on every image's title.

At every DDIM step, **no attempt is made to track a single query id across steps** --
independently pick the highest-IoU box at that step (lesson from building the DiffusionDet
notebook: the model can change its "target" between steps, and hard-tracking one id creates
a false impression that the box wandered off).

In [ ]:
@torch.no_grad()
def ddim_trajectory_for_image(idx, caption):
    image_t, target = dataset_val[idx]
    samples = utils.nested_tensor_from_tensor_list([image_t]).to(device)

    # re-create the first 3 lines of DiffuGroundingDINO.forward() to get (enc, tokenized)
    text_dict, tokenized = model._encode_text([caption], device)
    srcs, masks, poss = model._prepare_image_features(samples)
    enc = model.transformer.encode(srcs, masks, poss, text_dict)

    diffusion = model.diffusion
    batch_size = enc.batch_size
    x = diffusion.init_latent(batch_size, model.num_queries, device)

    # frame 0: the raw latent noise, mapped into box space by the clamp alone -- never
    # touches the decoder. This is the frame that is expected to hug the image edges.
    trace = [{"t": diffusion.num_timesteps - 1, "pred_boxes": diffusion.latent_to_boxes(x).clone(),
              "is_raw_noise": True}]

    for time, time_next in diffusion.ddim_time_pairs():
        timesteps = torch.full((batch_size,), time, device=device, dtype=torch.long)
        refpoints = diffusion.latent_to_refpoints(x)
        hs, references, _, _, _ = model.transformer.decode(enc, refpoint_embed=refpoints, timesteps=timesteps)
        _, outputs_coord = model._decode_predictions(hs, references, enc.text_dict)
        trace.append({"t": time, "pred_boxes": outputs_coord[-1].clone(), "is_raw_noise": False})

        x_start = diffusion.boxes_to_latent(outputs_coord[-1])
        x_start = torch.clamp(x_start, -diffusion.snr_scale, diffusion.snr_scale)
        pred_noise = diffusion.predict_noise_from_start(x, timesteps, x_start)
        x = diffusion.ddim_step(x, x_start, pred_noise, time, time_next)

    return trace, target

def _boxes_to_pixel_xyxy(pred_boxes_normed_cxcywh, w, h):
    xyxy = box_cxcywh_to_xyxy(pred_boxes_normed_cxcywh)
    return xyxy.cpu().numpy() * np.array([w, h, w, h])

def _step_label(step, step_number):
    return "z_T (raw noise)" if step["is_raw_noise"] else f"step {step_number} (t={step['t']})"

def draw_filmstrip(trace, gt_box, gt_cls_name, caption, max_w=PANEL_W):
    panels, subtitles = [], []
    step_number = 0
    for step in trace:
        boxes_px = _boxes_to_pixel_xyxy(step["pred_boxes"][0], img_w, img_h)
        ious = _iou_matrix(gt_box[None, :], boxes_px)[0]
        best_idx = int(np.argmax(ious))
        best_iou = float(ious[best_idx])

        im, scale = _resize_and_scale(base_image.copy(), max_w)
        d = ImageDraw.Draw(im)
        for b in boxes_px[:DRAW_PROPOSALS]:
            d.rectangle(_scaled(b, scale), outline=(255, 255, 255), width=BG_WIDTH)
        d.rectangle(_scaled(gt_box, scale), outline=C_GT_HIT, width=GT_WIDTH)
        bs = _scaled(boxes_px[best_idx], scale)
        d.rectangle(bs, outline=(40, 210, 90), width=TRACK_WIDTH)
        _label(d, bs[0], bs[1], f"IoU={best_iou:.2f}", (40, 210, 90))
        panels.append(im)
        if not step["is_raw_noise"]:
            step_number += 1
        subtitles.append(_step_label(step, step_number))

    title = (f'caption: "{caption}"  |  GT category = {gt_cls_name}  |  '
             f'raw noise -> final box over {len(trace) - 1} DDIM steps')
    return _strip(panels, subtitles, title)

def draw_heatmap_row(trace, gt_boxes_xyxy, caption, max_w=PANEL_W):
    panels, subtitles = [], []
    step_number = 0
    for step in trace:
        boxes_px = _boxes_to_pixel_xyxy(step["pred_boxes"][0], img_w, img_h)
        im, scale = _resize_and_scale(base_image.copy(), max_w)
        overlay = Image.new("RGBA", im.size, (0, 0, 0, 0))
        od = ImageDraw.Draw(overlay)
        for box in boxes_px[:DRAW_PROPOSALS]:
            bs = _scaled(box, scale)
            cx, cy = (bs[0] + bs[2]) / 2, (bs[1] + bs[3]) / 2
            od.ellipse([cx - DOT_R, cy - DOT_R, cx + DOT_R, cy + DOT_R], fill=(255, 60, 60, 160))
        im = Image.alpha_composite(im.convert("RGBA"), overlay).convert("RGB")
        d = ImageDraw.Draw(im)
        for box in gt_boxes_xyxy:
            d.rectangle(_scaled(box, scale), outline=C_GT_HIT, width=GT_WIDTH)
        panels.append(im)
        if not step["is_raw_noise"]:
            step_number += 1
        subtitles.append(_step_label(step, step_number))

    title = (f'caption: "{caption}"  |  centers of {DRAW_PROPOSALS} sample queries per step  |  '
             f'{len(gt_boxes_xyxy)} GT of this category (blue)')
    return _strip(panels, subtitles, title)

os.makedirs(f"{VIZ}/diffusion_trace", exist_ok=True)
for idx in picked_indices:
    image_id = dataset_val.ids[idx]
    file_name = coco_api.loadImgs(image_id)[0]["file_name"]
    base_image = Image.open(os.path.join(VAL_IMAGES, file_name)).convert("RGB")

    _, target0 = dataset_val[idx]
    img_h, img_w = target0["orig_size"].tolist()
    gt_boxes_all = box_cxcywh_to_xyxy(target0["boxes"]).numpy() * np.array([img_w, img_h, img_w, img_h]) \
        if len(target0["boxes"]) else np.zeros((0, 4))
    gt_labels_all = target0["labels"].tolist()

    # unique categories in this image, keeping first-appearance order
    seen, unique_cats = set(), []
    for cid in gt_labels_all:
        if cid not in seen:
            seen.add(cid)
            unique_cats.append(cid)
    unique_cats = unique_cats[:MAX_CATEGORIES_PER_IMAGE_TRACE]

    for cat_id in unique_cats:
        cat_name = catid_to_name.get(cat_id, str(cat_id))
        caption = build_caption([cat_name])
        trace, _ = ddim_trajectory_for_image(idx, caption)

        idx_in_image = [i for i, c in enumerate(gt_labels_all) if c == cat_id]
        gt_box_first = gt_boxes_all[idx_in_image[0]]
        img_filmstrip = draw_filmstrip(trace, gt_box_first, cat_name, caption)
        img_filmstrip.save(f"{VIZ}/diffusion_trace/{image_id}_{cat_name}_filmstrip.jpg", quality=JPEG_QUALITY)

        img_heatmap = draw_heatmap_row(trace, gt_boxes_all[idx_in_image], caption)
        img_heatmap.save(f"{VIZ}/diffusion_trace/{image_id}_{cat_name}_heatmap.jpg", quality=JPEG_QUALITY)

print(f"saved diffusion trace images -> {VIZ}/diffusion_trace/  "
      f"({len(os.listdir(f'{VIZ}/diffusion_trace'))} images)")

## 12. View the diffusion trace right here

In [ ]:
for fname in sorted(os.listdir(f"{VIZ}/diffusion_trace")):
    print(fname)
    display(IPyImage(filename=f"{VIZ}/diffusion_trace/{fname}"))

## 13. Package for download

In [ ]:
df.to_csv(f"{VIZ}/epoch_metrics.csv")
diag_df.to_csv(f"{VIZ}/score_threshold_diagnosis.csv", index=False)

import shutil
zip_path = shutil.make_archive(f"{WORK}/diffu_gdino_viz", "zip", VIZ)
print("zipped:", zip_path)
!du -sh {VIZ}
!ls -la {VIZ}